# Model Training and Testing

This notebook is an example of how to instanciate the dataset and the model for training and testing.

The chosen model example is the DSCNN, but based on the available set of models, this notebook can be easily modified.

In [ ]:
%reset -f

In [ ]:
import sys
sys.path.append('../')
import pyrul
import pandas as pd

2025-09-25 12:34:43.789475: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-25 12:34:43.845183: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-09-25 12:34:46.161083: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


Importing the dtw module. When using in academic works please cite:
  T. Giorgino. Computing and Visualizing Dynamic Time Warping Alignments in R: The dtw Package.
  J. Stat. Soft., doi:10.18637/jss.v031.i07.



In [ ]:
IDX = 0 # random internal seed

In [ ]:
# Selecting CMAPSS data :
data_load_path = "../datasets/cmapss/train_FD001_fixed.txt"

data_save_path = "./myModel_data_processed"
data_metrics_path = "./"
model_path = "./myModel"
tmp_plot_path = "./tmp_plots/"

In [ ]:
SEED = pyrul.utils.seed.get_seed(IDX)
pyrul.utils.seed.set_seed(SEED)

In [ ]:
# Clearing tempory plots folder
pyrul.utils.file.clear_tmp_plots(tmp_plot_path)

## I. Data and Model Construction

As a first step, the desired model and the data are declared through two respective objects : a <b>RULModel</b> and a <b>DataLoader</b>. 

The <b>RULModel</b> contains the hyperparameters, preprocessing steps and the core prediction models. To be declared, it needs to know which dataset to use, and what hyperparameters to have.

The <b>Dataloader</b> supervises model operations on data. Based on the chosen dataset, it provides the raw data to the models, and guides the entire data building process. Notably, the k-fold cross validation to create the datasets is done via this object. It is also an assistant to the user, to help saving and loading different preprocessed datasets or model information.

In [ ]:
# Declaring the model and its hyperparameters
# Several attributes have default arguments that are not shown here (eg: FOLDS=10 for CMAPSS)
model = pyrul.models.dscnn.DSCNN_Model(
     DATASET_NAME = "cmapss_fd001",
     LEARNING_RATE = 1e-3,
     SEED=SEED)

# Declaring the dataloader
dl = pyrul.models.data.DataLoader(dataset="femto")

In [ ]:
# Building the training and validation datasets through k-fold
dl.build_train_val_data(model, data_path=data_load_path)

In [ ]:
# Saving the training and validation datasets
dl.save_datasets(model, data=["train", "val"], path=data_save_path)
dl.save_data_metrics(model, path=data_metrics_path)

In [ ]:
# Loading the training and validation datasets
dl.load_datasets(model, data=["train", "val"], path=data_save_path)
dl.load_data_metrics(model, path=data_metrics_path)

## II. Model Training and Saving

The Dataloader has stored preprocessed data inside the Model object, so only the fitting process remains.

In [ ]:
# Fitting each model to their training dataset
model.fit_model()

In [ ]:
# Saving the fitted models
model.save_models(path=model_path)

## III. Test Data

Once the training and validation datasets have been made, the test data can also be constructed.
The test data is actually divided into a <b>"normal" test</b> dataset and a <b>"full" test</b> dataset. They correspond to the same testing units of the original dataset, but the "full" test dataset contains information of the entire incomplete run-to-failure test trajectories, so RUL prediction can be carried out during all available time steps. On the opposite, the "normal" test dataset only contains the last information window used by the model to predict the RUL at the last time step of each test units.

Like the training and validation datasets, the test sets are built, saved and loaded with the Dataloader.

In [ ]:
# Building the test and full test dataset
data_test_load_path = "../datasets/cmapss/test_FD001_fixed.txt"

dl.build_test_data(model, data_path=data_test_load_path)

In [ ]:
# Saving the test and full test datasets
dl.save_datasets(model, data=["test", "test_full"], path=data_save_path)

In [ ]:
# Loading the test and full test datasets
dl.load_datasets(model, data=["test", "test_full"], path=data_save_path)

## IV. Model Evaluation

Finally, the <b>Evaluator</b> object computes the performance of the trained Model based on specified metrics. In addition, utility functions can help visualize model predictions per units.

In [ ]:
# Declaring the evaluator and its metrics
evaluator = pyrul.evaluation.Evaluator(metrics=["rmse", "mae", "scoring_cmapss"])

In [ ]:
# Computing the metrics
df_metrics = evaluator.get_metrics(model, data="val")

In [ ]:
val_avg_rmse = df_metrics['val_rmse'].mean()
val_avg_score = df_metrics['val_scoring_cmapss'].mean()
print("Avg RMSE on validation:", val_avg_rmse)
print("Avg scoring on validation:", val_avg_score)

In [ ]:
# Plotting the model prediction for specific units, with specific models
pyrul.models.viz.plot_pred(model, 
                            plot_indices=[0,3,8], 
                            fold_indices=[0,0,0],
                            model_indices=[0,0,0], 
                            data_name="val", 
                            save_path=tmp_plot_path)
# In the above example, the model of the 0-th fold predicts the RUL values of the 0-th, 3-rd and 8-th units of its validation set.